# Analyse des données énergétiques

Ce notebook contient l'analyse des données énergétiques normalisées et transformées.

## 1. Configuration et imports

In [1]:
import sys
import os
import json
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import probplot, fisher_exact, chi2_contingency, shapiro, t, normaltest, jarque_bera, anderson

from sklearn.model_selection import train_test_split, cross_val_predict, RepeatedKFold, cross_val_score, GridSearchCV, ParameterGrid, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.utils import resample
from xgboost import XGBRegressor
import lightgbm as lgb

import statsmodels.api as sm
from statsmodels.api import OLS, WLS, add_constant
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import outlier_test, OLSInfluence

import openai
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

from dotenv import load_dotenv

import pprint


## 2. Chargement des données

In [2]:
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root not in sys.path:
    sys.path.insert(0, root)

# 2) importer la fonction
from data_loader import load_data

# 3) l’utiliser
df_converted = load_data()
df_converted.head()

,f_relative_compactness,f_surface_area,f_wall_area,f_roof_area,f_overall_height,f_orientation,f_glazing_area,f_glazing_area_distribution,l_heating_load,l_cooling_load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


In [3]:
# Liste des colonnes à supprimer
cols_to_drop = [
    'f_surface_area',
    'f_roof_area',
    'f_glazing_area_distribution'
]

# Ne supprimer que celles qui existent dans le DataFrame
existing_cols = [c for c in cols_to_drop if c in df_converted.columns]

# Suppression des colonnes (inplace)
df_converted.drop(columns=existing_cols, inplace=True)
df_selected = df_converted


## 4. Définitions des fonctions de transformation

In [4]:
# vos autres transform et inverse-transform
TRANSFORM_FNS = {
    'log':    np.log,
}

INV_TRANSFORM_FNS = {
    'log':    np.exp,
}

def apply_transform(series: pd.Series, method: str):
    """
    Retourne serie_t pour les autres méthodes
    """
    fn = TRANSFORM_FNS.get(method)
    if fn is None:
        raise ValueError(f"Transformation inconnue: {method}")
    return fn(series.values)

def inverse_transform(y, method: str):
    """
    Applique l'inverse de la transformation.
    """
    fn = INV_TRANSFORM_FNS.get(method)
    if fn is None:
        raise ValueError(f"Transformation inverse inconnue: {method}")
    return fn(y)

dict_l_transfo = {"l_cooling_load":"log",
                  "l_heating_load":"log"}
label_transformer = FunctionTransformer(
    func=apply_transform,
    inverse_func=inverse_transform,
    validate=False
)


## 5. Transformation des labels

### Objectif
Transformer les labels.


In [5]:
print("=" * 80)
print(f"Transformation des labels")
print("=" * 80)


# Copie du DataFrame pour stocker les labels transformés
df_transformed = df_selected.copy()

for label,method in dict_l_transfo.items():
    if method != "none":
        print(f"Transformation de {label} avec {method}")
        try:
            series_t = apply_transform(df_selected[label], method)
    
            # On remplace la colonne transformée
            df_transformed[label+"_orig"] = df_transformed[label]
            df_transformed[label] = series_t
        except Exception as e:
            print(f"⚠ Erreur pendant la transformation {method} de {label} : {e}")

df_transformed

Transformation des labels
Transformation de l_cooling_load avec log
Transformation de l_heating_load avec log


,f_relative_compactness,f_wall_area,f_overall_height,f_orientation,f_glazing_area,l_heating_load,l_cooling_load,l_cooling_load_orig,l_heating_load_orig
0,0.98,294.0,7.0,2,0.0,2.744061,3.060115,21.33,15.55
1,0.98,294.0,7.0,3,0.0,2.744061,3.060115,21.33,15.55
2,0.98,294.0,7.0,4,0.0,2.744061,3.060115,21.33,15.55
3,0.98,294.0,7.0,5,0.0,2.744061,3.060115,21.33,15.55
4,0.90,318.5,7.0,2,0.0,3.036874,3.342155,28.28,20.84
...,...,...,...,...,...,...,...,...,...
763,0.64,343.0,3.5,5,0.4,2.883683,3.063391,21.40,17.88
764,0.62,367.5,3.5,2,0.4,2.805782,2.826129,16.88,16.54
765,0.62,367.5,3.5,3,0.4,2.799717,2.839663,17.11,16.44
766,0.62,367.5,3.5,4,0.4,2.802148,2.810005,16.61,16.48


## 6. Fonction entraînement

In [6]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [20]:
import inspect
from pathlib import Path

# Support XGBoost silence
try:
    import xgboost as xgb
    xgb.set_config(verbosity=0)
except ImportError:
    xgb = None

# Support LightGBM
try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

warnings.filterwarnings("ignore", category=UserWarning)


def _manual_cv_lgbm(X, y, params, cv, random_state, early_stopping_rounds):
    """
    Réalise une CV manuelle avec early stopping pour un jeu de params LightGBM.
    Retourne la RMSE moyenne.
    """
    rmses = []
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        # Slice compatible Series ou ndarray
        y_tr = y[train_idx] if isinstance(y, np.ndarray) else y.iloc[train_idx]
        y_val = y[val_idx]   if isinstance(y, np.ndarray) else y.iloc[val_idx]

        m = LGBMRegressor(
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
            verbosity=-1,
            force_col_wise=True,
            **params
        )
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False),lgb.log_evaluation(0)],
            
        )        
        y_pred = m.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmses)


def train_models(
    df,
    model_cls,
    param_grid,
    apply_transform,
    inverse_transform,
    test_size=0.2,
    cv=5,
    random_state=42,
    apply_lgbm_adjustments=True,
    save=False
):
    results = {}
    save_dir = Path('saved_models')
    save_dir.mkdir(exist_ok=True)

    orig_labels = [c for c in df.columns if c.startswith('l_') and c.endswith('_orig')]

    for label_orig in orig_labels:
        label = label_orig[:-5]

        # 1) Transformation y
        trafo = dict_l_transfo[label]
        y_orig = df[label_orig]
        y_trf = apply_transform(y_orig, trafo)
        lmbda = None

        # 2) Préparation X
        feat_cols = [c for c in df.columns if c.startswith('f_')]
        X = df[feat_cols]
        numeric_feats = [c for c in feat_cols if pd.api.types.is_numeric_dtype(df[c])]
        categorical_feats = [c for c in feat_cols if not pd.api.types.is_numeric_dtype(df[c])]
        preproc = ColumnTransformer(
            transformers=[
                (
                    'num',
                    StandardScaler().set_output(transform="pandas"),
                    numeric_feats
                ),
                (
                    'cat',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
                        .set_output(transform="pandas"),
                    categorical_feats
                )
            ],
            remainder='drop',
            verbose_feature_names_out=False
        ).set_output(transform="pandas")
        # 3) Split
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y_trf,
            test_size=test_size,
            random_state=random_state
        )
        # 4) Initialisation dynamique
        init_params = inspect.signature(model_cls.__init__).parameters
        model_kwargs = {}
        
        # random_state & n_jobs pour tous
        if 'random_state' in init_params:
            model_kwargs['random_state'] = random_state
        if 'n_jobs' in init_params:
            model_kwargs['n_jobs'] = -1
        
        # verbose/verbosity pour LGBM seulement
        if LGBMRegressor and issubclass(model_cls, LGBMRegressor):
            if 'verbose' in init_params:
                model_kwargs['verbose'] = -1
            if 'verbosity' in init_params:
                model_kwargs['verbosity'] = -1
        if xgb and hasattr(xgb.XGBRegressor, '__init__') and issubclass(model_cls, xgb.XGBRegressor):
            # XGBRegressor accepte 'verbosity', pas 'verbose'
            if 'verbosity' in init_params:
                model_kwargs['verbosity'] = 0

        # 5) Cas LightGBM spécifique. On ne va pas utiliser la pipeline classique ni la gridsearch classique 
        #car ça empêche de faire de l'early stopping et rend l'entraînement extrêmement long.
        #On va donc coder chaque étape.
        if (apply_lgbm_adjustments
                and LGBMRegressor
                and issubclass(model_cls, LGBMRegressor)):

            # a) Préparer la grille et transformer X. On supprime le max depth = -1 qui peut pénaliser et fausser le LightGBM
            
            local_grid = param_grid.copy()
            if 'model__max_depth' in local_grid:
                vals = [d for d in local_grid['model__max_depth'] if d != -1]
                local_grid['model__max_depth'] = vals or local_grid['model__max_depth']
            plain_grid = {k.replace('model__', ''): v for k, v in local_grid.items()}

            preproc.fit(X_tr)
            X_tr_t = preproc.transform(X_tr)
            X_te_t = preproc.transform(X_te)

            # Récupérer noms de colonnes
            #cols_num = preproc.named_transformers_['num'].get_feature_names_out(numeric_feats)
            #cols_cat = preproc.named_transformers_['cat'].get_feature_names_out(categorical_feats)
            #feat_names = np.concatenate([cols_num, cols_cat])
            #X_tr_t = pd.DataFrame(X_tr_t, columns=feat_names, index=X_tr.index)
            #X_te_t = pd.DataFrame(X_te_t, columns=feat_names, index=X_te.index)

            # b) CV manuelle
            best_mean = np.inf
            best_params = None
            for params in ParameterGrid(plain_grid):
                mean_rmse = _manual_cv_lgbm(
                    X_tr_t, y_tr,
                    params,
                    cv=cv,
                    random_state=random_state,
                    early_stopping_rounds=10
                )
                if mean_rmse < best_mean:
                    best_mean = mean_rmse
                    best_params = params

            cv_rmse_trf = best_mean

            # c) Entraînement final
            final_model = model_cls(**{**model_kwargs, **best_params})
            final_model.fit(
                X_tr_t, y_tr,
                eval_set=[(X_te_t, y_te)],
                callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False),lgb.log_evaluation(0)]
            )
            best_pipe = Pipeline([('preproc', preproc), ('model', final_model)]).set_output(transform="pandas")

        else:
            # 6) Pipeline standard + GridSearchCV
            pipe = Pipeline([
                ('preproc', preproc),
                ('model', model_cls(**model_kwargs))
            ]).set_output(transform="pandas")
            gs = GridSearchCV(
                estimator=pipe,
                param_grid=param_grid,
                cv=cv,
                scoring='neg_root_mean_squared_error',
                n_jobs=-1,
                verbose=0
            )
            gs.fit(X_tr, y_tr)
            best_pipe = gs.best_estimator_
            best_params = gs.best_params_
            cv_rmse_trf = -gs.best_score_

        # 7) Évaluation sur le test set
        y_pred_trf = best_pipe.predict(X_te)
        if trafo != 'none':
            y_te_orig = inverse_transform(y_te, trafo)
            y_pred_orig = inverse_transform(y_pred_trf, trafo)
        else:
            y_te_orig, y_pred_orig = y_te, y_pred_trf

        test_rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))

        # 8) Sauvegarde
        if save:
            fname_base = f"{model_cls.__name__.lower()}_{label}"
            try:
                import joblib
                joblib.dump(best_pipe, save_dir / f"{fname_base}.joblib")
                print(f"[{label}] Pipeline saved: {fname_base}.joblib")
            except Exception as e:
                print(f"[{label}] Joblib save error: {e}")
    
            model = best_pipe.named_steps['model']

        # 9) Stockage résultats et prints finaux
        results[label] = {
            'model': best_pipe,
            'best_params': best_params,
            'cv_rmse_trf': cv_rmse_trf,
            'test_rmse_orig': test_rmse,
            'transformation': trafo,
            'lambda': lmbda
        }
        print(f"[{label}] Best params: {best_params}")
        print(f"CV RMSE(trf)={cv_rmse_trf:.4f}, Test RMSE(orig)={test_rmse:.4f}")

    return results


## 7. Entrainement XGBoost

In [21]:
# Exemple d'utilisation:
param_grid_xgb = {
#    'model__n_estimators':      [100, 200, 300],
    'model__n_estimators':      [50, 100, 300],
#    'model__max_depth':         [3, 6, 9],
    'model__max_depth':         [2, 3, 4],
#    'model__learning_rate':     [0.01, 0.1, 0.2],
    'model__learning_rate':     [0.1, 0.15, 0.2],
#    'model__subsample':         [0.6, 0.8, 1.0],
    'model__subsample':         [0.9,0.95, 1.0],
#    'model__colsample_bytree':  [0.6, 0.8, 1.0]
    'model__colsample_bytree':  [0.5,0.6, 0.7]
}
xgb_results = train_models(df_transformed,
                                XGBRegressor,
                                param_grid_xgb,
                                apply_transform,
                                inverse_transform)


[l_cooling_load] Best params: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.2, 'model__max_depth': 2, 'model__n_estimators': 300, 'model__subsample': 1.0}
CV RMSE(trf)=0.0504, Test RMSE(orig)=1.7471
[l_heating_load] Best params: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.15, 'model__max_depth': 3, 'model__n_estimators': 300, 'model__subsample': 1.0}
CV RMSE(trf)=0.0241, Test RMSE(orig)=0.5267


## 8. Entrainement LightGBM

In [22]:

from lightgbm import LGBMRegressor
param_grid_lgbm = {
#    'model__n_estimators':      [100, 200, 300],
    'model__n_estimators':      [50, 100, 300],
#    'model__max_depth':         [-1, 5, 10],
#    'model__max_depth':         [2, 5, 8],
    'model__max_depth':         [2, 5],
#    'model__learning_rate':     [0.01, 0.1, 0.2],
    'model__learning_rate':     [0.1, 0.15, 0.2],
#    'model__num_leaves':        [31, 63, 127],
#    'model__num_leaves':        [15, 31, 47],
    'model__num_leaves':        [12, 15, 22],
#    'model__subsample':         [0.6, 0.8, 1.0]
    'model__subsample':         [0.4, 0.6, 0.7]
}
lgbm_results = train_models(
    df_transformed,
    LGBMRegressor,
    param_grid_lgbm,
    apply_transform,
    inverse_transform,
    save=True
)

[l_cooling_load] Pipeline saved: lgbmregressor_l_cooling_load.joblib
[l_cooling_load] Best params: {'learning_rate': 0.15, 'max_depth': 2, 'n_estimators': 300, 'num_leaves': 12, 'subsample': 0.4}
CV RMSE(trf)=0.0516, Test RMSE(orig)=1.7164
[l_heating_load] Pipeline saved: lgbmregressor_l_heating_load.joblib
[l_heating_load] Best params: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 300, 'num_leaves': 12, 'subsample': 0.4}
CV RMSE(trf)=0.0273, Test RMSE(orig)=0.5227


In [10]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

In [11]:
# 1) Random Forest
param_grid_rf = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth':    [None, 10, 20],
    'model__max_features': ['sqrt', 'log2']
}
rf_results = train_models(
    df_transformed,
    RandomForestRegressor,
    param_grid_rf,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__max_depth': None, 'model__max_features': 'log2', 'model__n_estimators': 500}
CV RMSE(trf)=0.0611, Test RMSE(orig)=1.9780
[l_heating_load] Best params: {'model__max_depth': None, 'model__max_features': 'log2', 'model__n_estimators': 500}
CV RMSE(trf)=0.0448, Test RMSE(orig)=0.7551


In [12]:
# 2) Extra Trees
param_grid_et = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth':    [None, 10, 20],
    'model__max_features': ['sqrt', 'log2']
}
et_results = train_models(
    df_transformed,
    ExtraTreesRegressor,
    param_grid_et,
    apply_transform,
    inverse_transform
)


[l_cooling_load] Best params: {'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 100}
CV RMSE(trf)=0.0615, Test RMSE(orig)=1.9848
[l_heating_load] Best params: {'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 300}
CV RMSE(trf)=0.0493, Test RMSE(orig)=0.8450


In [13]:
# 3) Ridge Regression
param_grid_ridge = {
    'model__alpha': [0.01, 0.1, 1, 10]
}
ridge_results = train_models(
    df_transformed,
    Ridge,
    param_grid_ridge,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__alpha': 1}
CV RMSE(trf)=0.1131, Test RMSE(orig)=3.2075
[l_heating_load] Best params: {'model__alpha': 1}
CV RMSE(trf)=0.1221, Test RMSE(orig)=3.1696


In [14]:
# 4) Lasso Regression
param_grid_lasso = {
    'model__alpha': [0.01, 0.1, 1, 10]
}
lasso_results = train_models(
    df_transformed,
    Lasso,
    param_grid_lasso,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__alpha': 0.01}
CV RMSE(trf)=0.1149, Test RMSE(orig)=3.3523
[l_heating_load] Best params: {'model__alpha': 0.01}
CV RMSE(trf)=0.1225, Test RMSE(orig)=3.1644


In [15]:
# 5) ElasticNet
param_grid_enet = {
    'model__alpha': [0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.0, 0.5, 1.0]
}
enet_results = train_models(
    df_transformed,
    ElasticNet,
    param_grid_enet,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.0}
CV RMSE(trf)=0.1135, Test RMSE(orig)=3.2654


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.376e+00, tolerance: 7.595e-03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.534e+00, tolerance: 7.607e-03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_mode

[l_heating_load] Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.5}
CV RMSE(trf)=0.1219, Test RMSE(orig)=3.1706


In [16]:
# 6) Support Vector Regression
param_grid_svr = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto']
}
svr_results = train_models(
    df_transformed,
    SVR,
    param_grid_svr,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__C': 10, 'model__gamma': 'scale'}
CV RMSE(trf)=0.0816, Test RMSE(orig)=2.3032
[l_heating_load] Best params: {'model__C': 10, 'model__gamma': 'scale'}
CV RMSE(trf)=0.0789, Test RMSE(orig)=1.9109


In [17]:
# 7) K-Nearest Neighbors
param_grid_knn = {
    'model__n_neighbors': [3, 5, 10,20,30],
    'model__weights': ['uniform', 'distance']
}
knn_results = train_models(
    df_transformed,
    KNeighborsRegressor,
    param_grid_knn,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__n_neighbors': 3, 'model__weights': 'distance'}
CV RMSE(trf)=0.0717, Test RMSE(orig)=2.0928
[l_heating_load] Best params: {'model__n_neighbors': 3, 'model__weights': 'distance'}
CV RMSE(trf)=0.0876, Test RMSE(orig)=1.2918


In [18]:
# 8) Multi-layer Perceptron
param_grid_mlp = {
    'model__hidden_layer_sizes': [(50,), (100, 50), (100, 100, 50)],
    'model__alpha': [1e-4, 1e-3],
    'model__learning_rate_init': [1e-3, 1e-4]
}
mlp_results = train_models(
    df_transformed,
    MLPRegressor,
    param_grid_mlp,
    apply_transform,
    inverse_transform
)

/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

[l_cooling_load] Best params: {'model__alpha': 0.001, 'model__hidden_layer_sizes': (100, 100, 50), 'model__learning_rate_init': 0.001}
CV RMSE(trf)=0.1089, Test RMSE(orig)=2.5409


/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptro

[l_heating_load] Best params: {'model__alpha': 0.001, 'model__hidden_layer_sizes': (100, 100, 50), 'model__learning_rate_init': 0.001}
CV RMSE(trf)=0.1004, Test RMSE(orig)=2.0583


In [19]:
# 9) Gaussian Process Regression
param_grid_gpr = {
    'model__alpha': [1e-10, 1e-5, 1e-2]
}
gpr_results = train_models(
    df_transformed,
    GaussianProcessRegressor,
    param_grid_gpr,
    apply_transform,
    inverse_transform
)

[l_cooling_load] Best params: {'model__alpha': 1e-10}
CV RMSE(trf)=0.0670, Test RMSE(orig)=2.0052
[l_heating_load] Best params: {'model__alpha': 1e-10}
CV RMSE(trf)=0.0416, Test RMSE(orig)=0.6936
